# 01 · Your first LLM call

**AI Engineering Bootcamp · Module 1**

In this notebook you will:

1. Call an LLM from Python using the OpenAI SDK
2. Read the answer, the token usage and the latency
3. Send a conversation with system, user and assistant messages
4. See what temperature does
5. Switch from Gemini to Groq by changing two lines
6. Compare a weak prompt with a strong prompt

**Before you start**

- Open this folder in VS Code and select your `.venv` as the kernel (top right, *Select Kernel*).
- Copy `.env.example` to `.env` and paste in your own keys.
- Run a cell with **Shift + Enter**.

## Step 0 · Install packages (only needed once)

This installs into the same environment as the notebook's kernel.

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 1 · Load your keys

In [2]:
import os
import time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # reads the .env file in this folder

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")
GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")

# Never print the key itself, only whether it was found
print("Gemini key found:", bool(os.getenv("GEMINI_API_KEY")))
print("Groq key found:  ", bool(os.getenv("GROQ_API_KEY")))

Gemini key found: True
Groq key found:   True


## Step 2 · Make your first call

In [3]:
client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url=GEMINI_BASE_URL,
)

response = client.chat.completions.create(
    model=GEMINI_MODEL,
    messages=[
        {"role": "system", "content": "You are a concise tutor."},
        {"role": "user", "content": "Explain tokens in two lines."},
    ],
)

print(response.choices[0].message.content)

Tokens are the basic chunks of text—like words, parts of words, or punctuation—that AI models read and write. 
They are the fundamental units used to measure and process language data.


**If you get an error**

| Error | Usual cause |
|---|---|
| 401 | Wrong or missing key. Check `.env` and re-run Step 1. |
| 404 | Wrong model name. Copy the current name from the provider's model list. |
| 429 | Rate limit. Wait a minute, or switch to Groq (Step 6). |

## Step 3 · Read the response: text, tokens and time

In [4]:
start = time.perf_counter()
response = client.chat.completions.create(
    model=GEMINI_MODEL,
    messages=[{"role": "user", "content": "Give me 3 tips for writing clean Python."}],
)
latency = time.perf_counter() - start

print(response.choices[0].message.content)
print("-" * 40)
print("model:", response.model)
usage = response.usage
if usage:
    print("tokens in: ", usage.prompt_tokens)
    print("tokens out:", usage.completion_tokens)
print(f"latency:    {latency:.2f}s")

Here are 3 essential tips for writing clean, readable Python code:

### 1. Follow PEP 8 and Pythonic Conventions
PEP 8 is the official style guide for Python. Writing code that looks and feels standard makes it instantly recognizable and easier for other Python developers to read.
* **Use descriptive naming:** Use `snake_case` for functions and variables, `PascalCase` for classes, and `UPPER_CASE` for constants.
* **Avoid unnecessary boilerplate:** Don't write Java-style getters and setters when you can use properties, and avoid C-style loops when Python has built-in iterators.
* **Keep lines manageable:** Aim for a maximum line length of 79–88 characters and use blank lines sparingly to separate logical sections of your code.

### 2. Leverage List Comprehensions and Generator Expressions
Instead of writing multi-line `for` loops just to transform or filter data, use Python’s built-in comprehensions. They are faster, more concise, and often much easier to read at a glance.

* **Bad:**


## Step 4 · Messages and roles

The model has **no memory between calls**. A conversation is just a list of messages you send every time.
Here we keep the list, add the model's reply, then ask a follow-up question.

In [5]:
messages = [
    {"role": "system", "content": "You are a friendly Python tutor. Keep answers short."},
    {"role": "user", "content": "What is a list in Python?"},
]

reply = client.chat.completions.create(model=GEMINI_MODEL, messages=messages)
answer = reply.choices[0].message.content
print("Assistant:", answer)

# Add the reply and a follow-up question, then send the whole list again
messages.append({"role": "assistant", "content": answer})
messages.append({"role": "user", "content": "Show me an example."})

reply = client.chat.completions.create(model=GEMINI_MODEL, messages=messages)
print("\nAssistant:", reply.choices[0].message.content)
print("\nMessages sent this time:", len(messages))

Assistant: A list is an ordered collection of items in Python. Think of it like a shopping list! 

You create one using square brackets `[]`, and it can hold any mix of data types (like numbers, text, or even other lists).

Here is a quick example:
```python
my_list = ["apple", "banana", 3]}
```
Want to know how to add or get items from a list?

Assistant: Here is how you can create a list, add an item, and grab a specific item:

```python
# Create a list of fruits
fruits = ["apple", "banana", "cherry"]

# Add an item to the end using .append()
fruits.append("orange")

# Get the first item (Python starts counting at 0!)
print(fruits[0])  # Output: apple

print(fruits)     # Output: ['apple', 'banana', 'cherry', 'orange']
```

Want to try writing one yourself, or learn something else about lists?

Messages sent this time: 4


## Step 5 · Temperature

Lower temperature gives more focused, repeatable answers. Higher temperature gives more varied ones.

In [6]:
question = [{"role": "user", "content": "Suggest a name for a study-planner app. Reply with the name only."}]

for temp in [0.2, 0.2, 0.9, 0.9]:
    r = client.chat.completions.create(model=GEMINI_MODEL, messages=question, temperature=temp)
    print(f"temperature {temp}: {r.choices[0].message.content.strip()}")

temperature 0.2: StudyFlow
temperature 0.2: StudyFlow
temperature 0.9: Schola
temperature 0.9: Planify


## Step 6 · Switch provider: Gemini to Groq

Same code. Only the key, `base_url` and model change.

In [7]:
groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url=GROQ_BASE_URL,
)

start = time.perf_counter()
r = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[{"role": "user", "content": "Explain tokens in two lines."}],
)
print(r.choices[0].message.content)
print(f"\nGroq latency: {time.perf_counter() - start:.2f}s")

Tokens are the smallest pieces of text that a language model processes—often sub‑word units derived from a vocabulary of common patterns.  
They allow the model to represent any word efficiently while limiting the overall input size.

Groq latency: 0.66s


## Step 7 · Weak prompt vs strong prompt

In [9]:
text = """Git is a version control system. It tracks changes to files over time,
lets several people work on the same project, and makes it possible to go back
to an earlier version when something breaks."""

weak = "Summarize this."
strong = (
    "Summarize the text below in 3 bullet points for a beginner. "
    "Use simple English. If the text is not about software, reply 'Off topic'."
)

for name, prompt in [("WEAK", weak), ("STRONG", strong)]:
    r = client.chat.completions.create(
        model=GEMINI_MODEL,
        messages=[{"role": "user", "content": f"{prompt}\n\n{text}"}],
    )
    print(f"--- {name} ---")
    print(r.choices[0].message.content, "\n")

--- WEAK ---
Git is a version control system that tracks file changes, enables collaboration, and allows users to revert to earlier versions when issues arise. 

--- STRONG ---
* Git is a tool that keeps track of changes you make to files over time.
* It lets multiple people work together on the same project easily.
* It helps you go back to an older version of your work if something goes wrong. 



## Try it yourself

1. Change the system message in Step 2 to *"You explain everything using cricket examples."* What changes?
2. In Step 4, ask a third follow-up question. How many tokens in does it use now, and why?
3. In Step 7, replace `text` with a paragraph that is **not** about software. Does the strong prompt say *Off topic*?

## Try it yourself — solutions

In [10]:
# 1. System message with cricket examples
r = client.chat.completions.create(
    model=GEMINI_MODEL,
    messages=[
        {"role": "system", "content": "You explain everything using cricket examples."},
        {"role": "user", "content": "Explain what a Python list is."},
    ],
)
print(r.choices[0].message.content)

Imagine you are walking out to the middle of the pitch to open the batting, and you need to carry all your gear. Instead of carrying your helmet, bat, thigh guard, gloves, and box in separate hands, you put them all inside one **kit bag**. 

In Python, a **list** is exactly like that cricket kit bag. 

It is a single, ordered container that holds multiple items (we call them "elements") inside square brackets `[]`. 

Here is how a Python list works, explained over an innings:

### 1. Packing the Kit Bag (Creating a List)
Just like your kit bag holds all your equipment, a Python list can hold a collection of anything—runs scored, player names, or equipment.
```python
kit_bag = ["Bat", "Helmet", "Gloves", "Thigh Guard", "Box"]
```
You can even have a mixed bag—say, a list tracking a batsman's recent scores and their name:
```python
batsman_record = ["Virat Kohli", 183, 117, 73, True]
```

### 2. Guarding the Crease: Ordering and Indexing
In a cricket batting order, position is everything

In [11]:
# 2. Third follow-up — the whole conversation is resent, so tokens in grow
messages.append({"role": "assistant", "content": reply.choices[0].message.content})
messages.append({"role": "user", "content": "And how is a dictionary different?"})

reply = client.chat.completions.create(model=GEMINI_MODEL, messages=messages)
print("Assistant:", reply.choices[0].message.content)
print("\nMessages sent this time:", len(messages))
print("tokens in:", reply.usage.prompt_tokens, " <- higher, because every message is sent again on each call")

Assistant: Great question! While a list uses **numbers** (indices) to find items, a dictionary uses **keys** (like words in a real dictionary). 

Think of it like a contact card: you look up someone by their *name* (the key), not their position in line.

Here is an example:
```python
# A dictionary uses curly braces {} and key-value pairs
my_dog = {
    "name": "Buddy",
    "age": 3,
    "breed": "Golden Retriever"
}

# You grab items using the key instead of a number
print(my_dog["name"])  # Output: Buddy
```

See the difference? Lists are for ordered items; dictionaries are for labeled info!

Messages sent this time: 6
tokens in: 250  <- higher, because every message is sent again on each call


In [12]:
# 3. Off-topic paragraph — the strong prompt should reply 'Off topic'
text_offtopic = """The recipe for a good cup of chai: boil water with ginger, cardamom
and cinnamon, add tea leaves and milk, simmer for two minutes, then strain and sweeten."""

r = client.chat.completions.create(
    model=GEMINI_MODEL,
    messages=[{"role": "user", "content": f"{strong}\n\n{text_offtopic}"}],
)
print(r.choices[0].message.content)

Off topic
